# 1.Set-up

In [ ]:
!pip install ydata-profiling
!pip install catboost
!pip install xgboost
!pip install -U imbalanced-learn
!pip install -U scikit-learn
!pip install -U scikit-learn==1.3.2 imbalanced-learn==0.11.0
!pip install lightgbm
!pip install keras
!pip install tensorflow
!pip install scikit-learn==0.24
!pip install lazypredict

In [ ]:
import pandas as pd
import sys
import ast
#from ydata_profiling import ProfileReport
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import random
#from google.colab import files



from random import shuffle
from sklearn import preprocessing
from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split,cross_val_score, learning_curve
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier,GradientBoostingClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.tree import ExtraTreeClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder
from scipy.stats import norm, skew
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from keras import models, layers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


# Tensforflow libraries
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from tensorflow.keras.optimizers import Adam
from keras.layers import Flatten, Dense, Dropout, BatchNormalization
from keras.layers import Conv1D, MaxPool1D
%matplotlib inline


warnings.filterwarnings('ignore')

# 2.Data Loading

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
Dataset = pd.read_csv("data_Features (2).csv", on_bad_lines='skip', encoding='latin1')
Dataset.head()


# 3.Data Analysis

In [ ]:
Dataset.info()

In [ ]:
Dataset.nunique()

In [ ]:
Dataset.describe()

# 4.Feature engineering and data cleaning

In [ ]:
# Get the number of rows and columns
num_rows, num_columns = Dataset.shape

# Print the results
print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_columns}")

### Checking for Null values

In [ ]:
print("Number of NULL values:\n", Dataset.isnull().sum())

### Checking for Duplicate Rows

In [ ]:
print("Number of Duplicate Rows:", Dataset.duplicated().sum())

In [ ]:
 # Dataset.drop_duplicates(inplace=True)
 # print("Number of Duplicate Rows after Removal:", Dataset.duplicated().sum())
 # Drop non-numeric column like domain if present
if "domain" in Dataset.columns:
    Dataset = Dataset.drop("domain", axis=1)

# Convert all columns to numeric, force errors → NaN
Dataset = Dataset.apply(pd.to_numeric, errors="coerce")

# Drop rows with NaN
Dataset = Dataset.dropna()

### Check outliers

In [ ]:
numeric_columns = Dataset.select_dtypes(include=['number'])

# Extract column names
numeric_column_names = numeric_columns.columns.tolist()

# Print the names of numerical columns
print("Numerical Column Names:")
print(numeric_column_names)

In [ ]:
# Create separate boxplots for each numerical column
plt.figure(figsize=(20, 15))

for i, column in enumerate(numeric_column_names):
    plt.subplot(6, 6, i + 1)  # Adjust the subplot grid as needed
    sns.boxplot(x=column, data=Dataset, palette="Set2")
    plt.title(f"Boxplot of {column}")

plt.tight_layout()
plt.show()

In [ ]:
# Create separate violin plots for each numerical column
plt.figure(figsize=(20, 15))

for i, column in enumerate(numeric_column_names):
    plt.subplot(6, 6, i + 1)  # Adjust the subplot grid as needed
    sns.violinplot(x=column, data=Dataset, palette="Set2", inner="quartile")
    plt.title(f"Violin Plot of {column}")

plt.tight_layout()
plt.show()

## EDA

### Distribution graphs (Histograms)

In [ ]:
# Create separate histograms for each numerical column
plt.figure(figsize=(20, 15))

for i, column in enumerate(numeric_column_names):
    plt.subplot(6, 6, i + 1)  # Adjust the subplot grid as needed
    plt.hist(Dataset[column], bins=30, color='skyblue', edgecolor='black')
    plt.title(f"Histogram of {column}")

plt.tight_layout()
plt.show()

### Correlation matrix

In [ ]:
# Selecting only numerical columns
numeric_columns = Dataset[numeric_column_names]

# Calculate the correlation matrix
correlation_matrix = numeric_columns.corr()

# Plotting the correlation matrix using a heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=.5)
plt.title("Correlation Matrix")
plt.show()

### Balance or Non-Balance?
Number of Phishing(Attack) and Benign(Not-Attack)

In [ ]:
# Assuming S_dataset is your DataFrame
attack_counts = Dataset['label'].value_counts()

# Create a figure and a 1x2 subplot grid
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

# Bar plot on the first subplot
bars = axes[0].bar(attack_counts.index, attack_counts.values, color=['skyblue', 'lightcoral'])
axes[0].set_xlabel('label')
axes[0].set_ylabel('Count')
axes[0].set_title('Data Balance')

# Add count labels on top of each bar
for bar, count in zip(bars, attack_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(count),
                 ha='center', va='bottom')

# Pie chart on the second subplot
Dataset['label'].value_counts().plot.pie(explode=[0, 0.1], autopct='%1.2f%%', colors=['skyblue', 'lightcoral'], ax=axes[1])
axes[1].set_xlabel('label')
axes[1].set_ylabel('')  # Remove y-axis label for pie chart
axes[1].set_title('Data Balance')

# Adjust layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
# Extract features (X) and target variable (y)
X = Dataset.drop("label", axis=1) # X includes all columns except 'Type'
y = Dataset["label"]  # Target variable is 'Type'


### Identifying Features with Overwhelmingly Repeated Maximum Values

In [ ]:
def identify_repeated_max(df):
    cols_to_drop = []  # Initialize a list to store column names to be dropped

    cols = df.columns.values
    for col in cols:
        max_count = df[col].value_counts().max()
        total = len(df)
        max_percentage = (max_count / total) * 100

        if max_percentage > 90:  # Check if repetition percentage is over 97%
            print(f"Feature '{col}' has the largest value repeated more than 70% of the time.")
            print(f"Largest value: {df[col].value_counts().idxmax()}")
            print(f"Repetition percentage: {max_percentage:.2f}%")
            print()
            cols_to_drop.append(col)  # Add the column to the list of columns to be dropped

    return cols_to_drop  # Return the list of column names to be dropped

In [ ]:
# Call the function and get the list of columns to be dropped
columns_to_drop = identify_repeated_max(X)

print("Columns to be dropped:", columns_to_drop)

In [ ]:
cols_to_drop = identify_repeated_max(X)

X1 = X.drop(cols_to_drop, axis=1)

In [ ]:
# Get the number of rows and columns
num_rows, num_columns = X1.shape

# Print the results
print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_columns}")

### OverSampling
to make the data balance

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE only to the training set
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
# Now, X_train_resampled and y_train_resampled contain the resampled data with synthetic samples for the minority class


In [ ]:
y_train_resampled

### Balance or Non-Balance?
Number of Phishing(Attack) and Benign(Not-Attack)

In [ ]:
# Assuming S_dataset is your DataFrame
attack_counts = y_train_resampled.value_counts()

# Create a figure and a 1x2 subplot grid
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

# Bar plot on the first subplot
bars = axes[0].bar(attack_counts.index, attack_counts.values, color=['skyblue', 'lightcoral'])
axes[0].set_xlabel('label')
axes[0].set_ylabel('Count')
axes[0].set_title('Data Balance')

# Add count labels on top of each bar
for bar, count in zip(bars, attack_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(count),
                 ha='center', va='bottom')

# Pie chart on the second subplot
y_train_resampled.value_counts().plot.pie(explode=[0, 0.1], autopct='%1.2f%%', colors=['skyblue', 'lightcoral'], ax=axes[1])
axes[1].set_xlabel('label')
axes[1].set_ylabel('')  # Remove y-axis label for pie chart
axes[1].set_title('Data Balance')

# Adjust layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
print("Number of Duplicate Rows:", Dataset.duplicated().sum())

# 5.Modeling

### Apply LazyClassifier

In [ ]:
# Disable GPU and clear memory
import os, gc, psutil
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Force CPU-only mode

# Function to show RAM usage
def show_ram_usage():
    print(f"RAM used: {psutil.virtual_memory().percent}%")

gc.collect()
show_ram_usage()

# Import LazyPredict and dependencies
!pip install lazypredict --quiet
from lazypredict.Supervised import LazyClassifier

# (Optional) Downsample large dataset for stability
# If your training data has >20k samples, reduce it temporarily.
if len(X_train_resampled) > 20000:
    import pandas as pd
    import numpy as np
    sample_idx = np.random.choice(len(X_train_resampled), 20000, replace=False)
    X_train_small = X_train_resampled.iloc[sample_idx]
    y_train_small = y_train_resampled.iloc[sample_idx]
else:
    X_train_small = X_train_resampled
    y_train_small = y_train_resampled

# Same for test set (optional)
if len(X_test) > 5000:
    test_idx = np.random.choice(len(X_test), 5000, replace=False)
    X_test_small = X_test.iloc[test_idx]
    y_test_small = y_test.iloc[test_idx]
else:
    X_test_small = X_test
    y_test_small = y_test

# Initialize LazyClassifier with safe settings
clf = LazyClassifier(
    predictions=False,   # Don’t store all predictions → saves memory
    custom_metric=None,  # Default metrics only
)

# Fit and evaluate
models, _ = clf.fit(X_train_small, X_test_small, y_train_small, y_test_small)

# Display results
import pandas as pd
pd.set_option('display.max_rows', None)
print(models)

# Check memory usage after fitting
gc.collect()
show_ram_usage()


In [ ]:
models

## Classfication Models

In [ ]:
def bias_variance(clf, x_train, x_test, y_train, y_test):
 #label_encoder object knows how to understand word labels.
        label_encoder = preprocessing.LabelEncoder()
        X_train_copy = np.copy(x_train)
        X_test_copy = np.copy(x_test)
        y_train_copy = np.copy(y_train)
        y_test_copy = np.copy(y_test)

        # Predict the labels for training and test data
        y_train_pred = clf.predict(x_train)
        y_test_pred = clf.predict(x_test)

        # Encode the true and predicted labels
        y_train_encoded = label_encoder.fit_transform(y_train_copy)
        y_test_encoded = label_encoder.fit_transform(y_test_copy)
        y_train_pred_encoded = label_encoder.transform(y_train_pred)
        y_test_pred_encoded = label_encoder.transform(y_test_pred)

        # Calculate the average bias
        avg_bias = np.mean((y_train_encoded - y_train_pred_encoded) ** 2)

        # Calculate the average variance
        avg_var = np.mean(np.var(y_train_pred_encoded, axis=0))

        print('Average bias: %.3f' % avg_bias)
        print('Average variance: %.3f' % avg_var)

In [ ]:
pastel_palette = sns.color_palette('pastel')
model_results = {}
classifiers_scores = {}

def bias_variance(clf, X_train, X_test, y_train, y_test):
    # Implementation of bias-variance decomposition is needed here
    pass

def Classifiers(X_train, y_train, X_test, y_test):
    val_results = {}

    try:
        bagging = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42))
    except TypeError:
        bagging = BaggingClassifier(base_estimator=DecisionTreeClassifier(random_state=42))

    classifiers = {
        "Logistic Regression": LogisticRegression(),
        #"Support Vector Machine SVM": SVC(C=100, gamma=0.002),
        #"Support Vector Machine SVM (RBF)": SVC(C=100, gamma=0.002, kernel='rbf'),
        "Decision Tree Classifier": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "AdaBoost": AdaBoostClassifier(n_estimators=50, random_state=42),
        "XGB Extreme X Gradient Boosting": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
        "Catboost": CatBoostClassifier(iterations=50, random_state=42, verbose=False),
        "LGBM": LGBMClassifier(random_state=42),
        "Stochastic Gradient Descent SGD": SGDClassifier(loss='modified_huber', random_state=42),
        "Gaussian Naive Bayes": GaussianNB(),
        "Bernoulli Naive Bayes": BernoulliNB(),
        "Extra Trees Classifier": ExtraTreesClassifier(n_estimators=100, random_state=42),
        "Extra Tree Classifier": ExtraTreeClassifier(random_state=42),
        "K Nearest Neighbors": KNeighborsClassifier(),
        "Bagging Classifier": bagging,
    }

    # Shuffle the data to ensure randomness in cross-validation
    # X_train, y_train = shuffle(X_train, y_train, random_state=42)
    f1_scores = {}
    clf_models = {}  # Dictionary to store trained models

    for clf_name, clf in classifiers.items():
        print(f"{clf_name}")
        print("------------------------------------------------------------------------------------------------------")

        val_accuracy = cross_val_score(clf, X_train, y_train, cv=10)
        val_results[clf_name] = val_accuracy
        print("Cross_validation Accuracy for", clf_name, ":\n", val_results[clf_name])
        print("------------------------------------------------------------------------------------------------------")

        clf_model = clf.fit(X_train, y_train)
        clf_models[clf_name] = clf_model  # Store the trained model
        y_test_pred = clf_model.predict(X_test)
        y_train_pred = clf_model.predict(X_train)

        # Calculate Average bias, Average variance
        bias_variance(clf, X_train, X_test, y_train, y_test)
        print("------------------------------------------------------------------------------------------------------")

        ##############################################################################################################
        # Confusion Matrix for Training and Testing
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)

        # Visualize the confusion matrices using Seaborn
        fig, axs = plt.subplots(ncols=2, figsize=(10, 5))

        sns.heatmap(cm_train, annot=True, cmap='Pastel2', ax=axs[0], fmt='g')
        axs[0].set_title(f"{clf_name} Training Confusion Matrix")
        axs[0].set_xlabel('Predicted Labels')
        axs[0].set_ylabel('Actual Labels')

        sns.heatmap(cm_test, annot=True, cmap='Pastel1', ax=axs[1], fmt='g')
        axs[1].set_title(f"{clf_name} Testing Confusion Matrix")
        axs[1].set_xlabel('Predicted Labels')
        axs[1].set_ylabel('Actual Labels')

        plt.tight_layout()
        plt.show()
        print("------------------------------------------------------------------------------------------------------")

        ##############################################################################################################
        # F1 Score for Training and Testing
        Train_F1 = f1_score(y_train, y_train_pred)
        Test_F1 = f1_score(y_test, y_test_pred)
        f1_scores[clf_name] = Test_F1

        model_results[clf_name] = Test_F1

        print("Train F1 Score is:", Train_F1)
        print("Test F1 Score is:", Test_F1)
        print("------------------------------------------------------------------------------------------------------")

        ##############################################################################################################
        # Classification Report for Training and Testing
        clf_report_Train = classification_report(y_train, y_train_pred)
        print(f"{clf_name} Training Classification Report:\n{clf_report_Train}")
        print("------------------------------------------------------------------------------------------------------")

        clf_report_Test = classification_report(y_test, y_test_pred)
        print(f"{clf_name} Testing Classification Report:\n{clf_report_Test}")
        print("------------------------------------------------------------------------------------------------------")

        ##############################################################################################################
        # Performance Metrics for Training and Testing
        acc_train = accuracy_score(y_train, y_train_pred)
        pre_train = precision_score(y_train, y_train_pred)
        recall_train = recall_score(y_train, y_train_pred)
        f1_train = f1_score(y_train, y_train_pred)

        # Create DataFrame for training metrics
        metrics_train = pd.DataFrame({'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 score'],
                                      'Value': [acc_train, pre_train, recall_train, f1_train]})

        acc_test = accuracy_score(y_test, y_test_pred)
        pre_test = precision_score(y_test, y_test_pred)
        recall_test = recall_score(y_test, y_test_pred)
        f1_test = f1_score(y_test, y_test_pred)

        # Create DataFrame for testing metrics
        metrics_test = pd.DataFrame({'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 score'],
                                     'Value': [acc_test, pre_test, recall_test, f1_test]})

        classifiers_scores[clf_name] = [acc_test, pre_test, recall_test, f1_test]

        fig, axs = plt.subplots(ncols=2, figsize=(10, 5))

        sns.barplot(x='Metric', y='Value', data=metrics_train, ax=axs[0], palette=pastel_palette)
        axs[0].set_title(f"{clf_name} Training Performance Metrics")

        sns.barplot(x='Metric', y='Value', data=metrics_test, ax=axs[1], palette=pastel_palette)
        axs[1].set_title(f"{clf_name} Testing Performance Metrics")

        plt.tight_layout()
        plt.show()

    # Return all classifiers along with their F1 scores
    return clf_models, classifiers


In [ ]:
# Assuming you have your data loaded into X_train, y_train, X_test, and y_test
models,classifiers = Classifiers(X_train_resampled, y_train_resampled, X_test, y_test)

#6. Apply Stacking Classifier and Soft Voting and compare the results with the champion model as a new approach in this project.

#### Calculate the TP , TN and F1_score(average='macro') for all models  

In [ ]:
models_results={}
models_predictions={}
for clf_name, clf in models.items():


    y_test_pred = clf.predict(X_test)
    f1_test = f1_score(y_test, y_test_pred, average='macro')
    ##############################################################################################################
    # Confusion Matrix for Training and Testing
    tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
    models_results[clf_name] = {'TP': tp, 'TN': tn, 'f1': f1_test, 'y_test_pred': y_test_pred}
    models_predictions[clf_name] = {'y_test_pred': y_test_pred}
    print(f"{clf_name} Results:\n", models_results[clf_name])
    print("------------------------------------------------------------------------------------------------------")
    print("------------------------------------------------------------------------------------------------------")

#### Find the Champion Model

In [ ]:
# Display the champion model based on F1 score
sorted_results_f1 = sorted(models_results.items(), key=lambda x: x[1]['f1'], reverse=True)
champion_model, champion_metrics = sorted_results_f1[0]

# Create a DataFrame for the champion model
df_champion = pd.DataFrame({'Model': [champion_model], 'F1_Score': [champion_metrics['f1']]})

# Display the DataFrame
print("\nChampion Model based on F1_Score:")
# print(df_champion)
df_champion

#### Sort the Models Results in Descending Order Based on TP

In [ ]:
# Display the results in descending order based on tp_test
sorted_results_tp = sorted(models_results.items(), key=lambda x: x[1]['TP'], reverse=True)

# Create a DataFrame from the results
df_tp = pd.DataFrame(sorted_results_tp, columns=['Model', 'Models Results in Descending Order based on TP'])

# If you want to add styling, you can use the following:
styled_table_tp = df_tp.style.background_gradient(cmap='Blues')
# display(styled_table_tp)
styled_table_tp

#### Sort the Models Results in Descending Order Based on TN

In [ ]:
# Assuming models_results is a dictionary where each entry is {model_name: {'TN': value, ...}}
sorted_results_tn = sorted(models_results.items(), key=lambda x: x[1]['TN'], reverse=True)

# Create a DataFrame from the results
df_tn = pd.DataFrame(sorted_results_tn, columns=['Model', 'Models Results in Descending Order based on TN'])

# If you want to add styling, you can use the following:
styled_table_tn = df_tn.style.background_gradient(cmap='Blues')
# display(styled_table_tn)
styled_table_tn

#### Sort the Models Results in Descending Order Based on F1_Score

In [ ]:
# Display the results in descending order based on f1 score
sorted_results_f1 = sorted(models_results.items(), key=lambda x: x[1]['f1'], reverse=True)

# Create a DataFrame from the results
df_f1 = pd.DataFrame(sorted_results_f1, columns=['Model', 'Models Results in Descending Order based on F1_Score'])

# If you want to add styling, you can use the following:
styled_table_f1 = df_f1.style.background_gradient(cmap='Blues')
# display(styled_table_f1)
styled_table_f1

#### Select the best two models in TP

In [ ]:
top_models_tp = sorted_results_tp[:5]
# Create a DataFrame from the results
df_tp = pd.DataFrame(top_models_tp, columns=['Model', 'Sort by Tp_Score'])
styled_table_tp = df_tp.style.background_gradient(cmap='Blues')
# display(styled_table_tp)
styled_table_tp

#### Select the best two models in TN

In [ ]:
top_models_tn = sorted_results_tn[:5]
# Create a DataFrame from the results
df_tn = pd.DataFrame(top_models_tn, columns=['Model', 'Sort by Tn_Score'])
styled_table_tn = df_tn.style.background_gradient(cmap='Blues')
# display(styled_table_tn)
styled_table_tn

#### Select the best two models in F1_Score

In [ ]:
top_models_f1 = sorted_results_f1[:5]
# Create a DataFrame from the results
df_f1 = pd.DataFrame(top_models_f1, columns=['Model', 'Sort by F1_Score'])
styled_table_f1 = df_f1.style.background_gradient(cmap='Blues')
# display(styled_table_f1)
styled_table_f1

# 7.Apply Fusion Classifier

#### Apply Stacking Classifier on the best two models in TP

In [ ]:
# Initialize the base models
base_models = [
    # ("Stochastic Gradient Descent SGD", SGDClassifier(loss='modified_huber', random_state=42)),
    ("XGB Extreme X Gradient Boosting", XGBClassifier(use_label_encoder=False)),
    ("LGBM", LGBMClassifier(random_state=42)),
]

#Initialize the stacking classifier with the meta-model
stacking_classifier1 = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(random_state=42)
)

# Fit the stacking classifier on the training data
stacking_classifier1.fit(X_train_resampled, y_train_resampled)

#### Apply Stacking Classifier on the best two models in TN

In [ ]:
# Initialize the base models
base_models = [
    ("Gradient Boosting", GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ("Gaussian Naive Bayes", GaussianNB())
    # ("Bagging Classifier", BaggingClassifier(base_estimator=DecisionTreeClassifier(random_state=42), n_estimators=100, random_state=42)),
]

#Initialize the stacking classifier with the meta-model
stacking_classifier = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(random_state=42)
)

# Fit the stacking classifier on the training data
stacking_classifier.fit(X_train_resampled, y_train_resampled)

#### Apply soft voting to the results of the Stacking Classifier for both TP and TN.

In [ ]:
# Create the ensemble model using soft voting
ensemble_classifier = VotingClassifier(estimators=[
    ("tn", stacking_classifier),
    ("tp", stacking_classifier1),
], voting='soft')

# Fit the ensemble model on the training data
ensemble_classifier.fit(X_train_resampled, y_train_resampled)

# Make predictions on the validation data
y_pred_ensemble = ensemble_classifier.predict(X_test)

In [ ]:
f1_test = f1_score(y_test, y_pred_ensemble, average='macro')
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_ensemble).ravel()
print(f"f1:{f1_test}")

#### Compare the results.

##### As shown here, the results of soft voting after selecting best of TP and TN are better than the champion model.

In [ ]:
# Create a DataFrame for the ensemble classifier
df_ensemble = pd.DataFrame({'Model': ['Ensemble Classifier'], 'F1_Score': [f1_test]})

# Display the DataFrame for the ensemble classifier
print("\nEnsemble Classifier based on F1_Score:")
print(df_ensemble)

# Compare the results in a single DataFrame
df_comparison = pd.concat([df_champion, df_ensemble], ignore_index=True)

# Display the comparison DataFrame
print("\nComparison of Champion Model and Ensemble Classifier:")
# print(df_comparison)
df_comparison

In [ ]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
